# 3.1 PostgreSQL Connection

In [1]:
import psycopg2
import pandas as pd

print("psycopg2:", psycopg2.__version__)

psycopg2: 2.9.11 (dt dec pq3 ext lo64)


In [ ]:
conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="postgres",
    user="postgres",
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

print("PostgreSQL connection successful")

PostgreSQL connection successful


In [3]:
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit


In [4]:
cursor.execute("SELECT current_database(), current_user;")
print(cursor.fetchone())

('postgres', 'postgres')


In [5]:
cursor.execute("""
    SELECT datname
    FROM pg_database
    WHERE datistemplate = false;
""")

cursor.fetchall()

[('postgres',), ('promotional_analytics',)]

# 3.2 Database Setup

In [ ]:
conn.rollback()
conn.autocommit = True

cursor.execute("""
    CREATE DATABASE promotional_analytics
    WITH OWNER = postgres;
""")

print("Database created successfully")

Database created successfully


In [7]:
conn.close()

conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="promotional_analytics",
    user="postgres",
    password="Saksham@3124"
)

conn.autocommit = True
cursor = conn.cursor()

print("Connected to promotional_analytics")

Connected to promotional_analytics


In [8]:
import pandas as pd
import os

data_path = "../Data"

for file in os.listdir(data_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_path, file))
        print(f"\n{file}")
        print(df.shape)
        print(df.columns.tolist())


campaign_desc.csv
(30, 4)
['DESCRIPTION', 'CAMPAIGN', 'START_DAY', 'END_DAY']

campaign_table.csv
(7208, 3)
['DESCRIPTION', 'household_key', 'CAMPAIGN']

causal_data.csv
(36786524, 5)
['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']

coupon.csv
(124548, 3)
['COUPON_UPC', 'PRODUCT_ID', 'CAMPAIGN']

coupon_redempt.csv
(2318, 4)
['household_key', 'DAY', 'COUPON_UPC', 'CAMPAIGN']

hh_demographic.csv
(801, 8)
['classification_1', 'classification_2', 'classification_3', 'HOMEOWNER_DESC', 'classification_5', 'classification_4', 'KID_CATEGORY_DESC', 'household_key']

product.csv
(92353, 7)
['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']

transaction_data.csv
(2595732, 12)
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']


In [9]:
product = pd.read_csv("../Data/product.csv")

print(product.shape)
print(product.columns.tolist())

(92353, 7)
['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']


# 3.3 Data Loading

In [10]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS transaction_data (
    household_key INTEGER,
    basket_id BIGINT,
    day INTEGER,
    product_id BIGINT,
    quantity INTEGER,
    sales_value NUMERIC(12,2),
    store_id INTEGER,
    retail_disc NUMERIC(12,2),
    trans_time INTEGER,
    week_no INTEGER,
    coupon_disc NUMERIC(12,2),
    coupon_match_disc NUMERIC(12,2)
);
""")

conn.commit()

print("transaction_data table created")

transaction_data table created


In [11]:
import os

cursor.execute("TRUNCATE TABLE transaction_data;")
conn.commit()

csv_path = os.path.abspath("../Data/transaction_data.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    cursor.copy_expert(
        """
        COPY transaction_data
        FROM STDIN
        WITH (FORMAT CSV, HEADER TRUE)
        """,
        f
    )

conn.commit()

print("transaction_data refreshed successfully")

transaction_data refreshed successfully


In [12]:
cursor.execute("SELECT COUNT(*) FROM transaction_data;")
print(cursor.fetchone())

(2595732,)


In [13]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS product (
    product_id BIGINT,
    manufacturer INTEGER,
    department VARCHAR(100),
    brand VARCHAR(100),
    commodity_desc VARCHAR(255),
    sub_commodity_desc VARCHAR(255),
    curr_size_of_product VARCHAR(100)
);
""")

conn.commit()

print("product table created")

product table created


In [14]:
import os

cursor.execute("TRUNCATE TABLE product;")

conn.commit()

csv_path = os.path.abspath("../Data/product.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    cursor.copy_expert(
        """
        COPY product
        FROM STDIN
        WITH (FORMAT CSV, HEADER TRUE)
        """,
        f
    )

conn.commit()

print("product refreshed successfully")

product refreshed successfully


In [15]:
query = """
SELECT
    (SELECT COUNT(*) FROM transaction_data) AS transaction_rows,
    (SELECT COUNT(*) FROM product) AS product_rows;
"""

load_check = pd.read_sql(query, conn)

load_check

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\71919010.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  load_check = pd.read_sql(query, conn)


,transaction_rows,product_rows
0,2595732,92353


In [16]:
cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS unique_products
    FROM product;
""")

print(cursor.fetchone())

(92353, 92353)


In [17]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS campaign_table (
    description VARCHAR(100),
    household_key INTEGER,
    campaign INTEGER
);

CREATE TABLE IF NOT EXISTS coupon_redempt (
    household_key INTEGER,
    day INTEGER,
    coupon_upc BIGINT,
    campaign INTEGER
);

CREATE TABLE IF NOT EXISTS campaign_desc (
    description VARCHAR(100),
    campaign INTEGER,
    start_day INTEGER,
    end_day INTEGER
);

CREATE TABLE IF NOT EXISTS hh_demographic (
    classification_1 VARCHAR(100),
    classification_2 VARCHAR(100),
    classification_3 VARCHAR(100),
    homeowner_desc VARCHAR(100),
    classification_5 VARCHAR(100),
    classification_4 VARCHAR(100),
    kid_category_desc VARCHAR(100),
    household_key INTEGER
);
""")

conn.commit()
print("All tables created")

All tables created


In [18]:
files = [
    ("campaign_table", "../Data/campaign_table.csv"),
    ("coupon_redempt", "../Data/coupon_redempt.csv"),
    ("campaign_desc", "../Data/campaign_desc.csv"),
    ("hh_demographic", "../Data/hh_demographic.csv")
]

for table, path in files:
    with open(os.path.abspath(path), "r", encoding="utf-8") as f:
        cursor.copy_expert(
            f"COPY {table} FROM STDIN WITH (FORMAT CSV, HEADER TRUE)",
            f
        )

conn.commit()
print("All datasets loaded successfully")


All datasets loaded successfully


# 3.4 SQL Validation & Analytics

## 3.4.1 Household Performance

In [19]:
tables = [
    "transaction_data",
    "product",
    "campaign_table",
    "coupon_redempt",
    "campaign_desc",
    "hh_demographic"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    print(f"{table}: {cursor.fetchone()[0]:,}")

transaction_data: 2,595,732
product: 92,353
campaign_table: 79,288
coupon_redempt: 25,498
campaign_desc: 330
hh_demographic: 8,811


In [20]:
query = """
SELECT
    household_key,
    SUM(sales_value) AS sales,
    COUNT(DISTINCT basket_id) AS transactions,
    SUM(quantity) AS quantity,
    ROUND(
        SUM(sales_value) / NULLIF(COUNT(DISTINCT basket_id), 0),
        2
    ) AS sales_per_transaction
FROM transaction_data
GROUP BY household_key
ORDER BY sales DESC;
"""

household_sql = pd.read_sql(query, conn)

household_sql.head(10)

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\364026027.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  household_sql = pd.read_sql(query, conn)


,household_key,sales,transactions,quantity,sales_per_transaction
0,1023,38319.79,603,4479917,63.55
1,1609,27859.68,412,2146715,67.62
2,2322,23646.92,323,992718,73.21
3,1453,21661.29,761,95743,28.46
4,2459,20671.50,971,488789,21.29
5,1430,20352.99,344,1741892,59.17
6,718,19299.86,599,869732,32.22
7,707,19194.42,498,1640193,38.54
8,1653,19153.75,541,1065654,35.40
9,1111,18894.72,321,11216,58.86


In [21]:
print(household_sql.shape)

(2500, 5)


In [22]:
household_sql.describe()

,household_key,sales,transactions,quantity,sales_per_transaction
count,2500.00000,2500.000000,2500.000000,2.500000e+03,2500.000000
mean,1250.50000,3222.985232,110.593600,1.042742e+05,31.620732
std,721.83216,3349.026076,115.669368,2.444561e+05,19.054951
min,1.00000,8.170000,1.000000,5.000000e+00,2.390000
25%,625.75000,970.740000,39.000000,7.700000e+02,18.327500
50%,1250.50000,2157.750000,79.000000,1.108450e+04,27.420000
75%,1875.25000,4413.320000,142.250000,8.764000e+04,40.547500
max,2500.00000,38319.790000,1300.000000,4.479917e+06,165.830000


In [23]:
household_sql[
    household_sql["household_key"] == 1023
]

,household_key,sales,transactions,quantity,sales_per_transaction
0,1023,38319.79,603,4479917,63.55


## 3.4.2 Department Performance

In [24]:
query = """
SELECT
    p.department,
    SUM(t.sales_value) AS sales,
    SUM(t.quantity) AS quantity,
    COUNT(DISTINCT t.basket_id) AS transactions,
    COUNT(DISTINCT t.household_key) AS households
FROM transaction_data t
JOIN product p
    ON t.product_id = p.product_id
GROUP BY p.department
ORDER BY sales DESC;
"""

department_sql = pd.read_sql(query, conn)

department_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\991283721.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  department_sql = pd.read_sql(query, conn)


,department,sales,quantity,transactions,households
0,GROCERY,4093814.14,2194762,215411,2500
1,DRUG GM,1055358.03,353844,118195,2491
2,PRODUCE,557452.11,319993,89026,2456
3,MEAT,548786.81,119113,53401,2351
4,KIOSK-GAS,544222.28,221254887,22056,1371
5,MEAT-PCKGD,412436.77,148148,58466,2392
6,DELI,260866.51,67026,35639,2248
7,PASTRY,121739.86,49820,30099,2281
8,MISC SALES TRAN,119960.04,36080860,5992,1393
9,NUTRITION,97669.04,43253,16820,1719


In [25]:
department_sql.head(10)

,department,sales,quantity,transactions,households
0,GROCERY,4093814.14,2194762,215411,2500
1,DRUG GM,1055358.03,353844,118195,2491
2,PRODUCE,557452.11,319993,89026,2456
3,MEAT,548786.81,119113,53401,2351
4,KIOSK-GAS,544222.28,221254887,22056,1371
5,MEAT-PCKGD,412436.77,148148,58466,2392
6,DELI,260866.51,67026,35639,2248
7,PASTRY,121739.86,49820,30099,2281
8,MISC SALES TRAN,119960.04,36080860,5992,1393
9,NUTRITION,97669.04,43253,16820,1719


In [26]:
query = """
SELECT SUM(sales_value) AS total_sales
FROM transaction_data;
"""

sql_total_sales = pd.read_sql(query, conn)

print(sql_total_sales)

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\3113738323.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_total_sales = pd.read_sql(query, conn)


   total_sales
0   8057463.08


## 3.4.3 Campaign Performance

In [27]:
query = """
SELECT
    campaign,
    COUNT(DISTINCT household_key) AS households_targeted
FROM campaign_table
GROUP BY campaign
ORDER BY households_targeted DESC;
"""

campaign_sql = pd.read_sql(query, conn)

campaign_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\2359061622.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  campaign_sql = pd.read_sql(query, conn)


,campaign,households_targeted
0,18,1133
1,13,1077
2,8,1076
3,30,361
4,26,332
5,22,276
6,20,244
7,14,224
8,11,214
9,17,202


In [28]:
query = """
SELECT
    campaign,
    COUNT(*) AS redemptions,
    COUNT(DISTINCT household_key) AS households_redeemed
FROM coupon_redempt
GROUP BY campaign
ORDER BY redemptions DESC;
"""

campaign_response_sql = pd.read_sql(query, conn)

campaign_response_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\1014880972.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  campaign_response_sql = pd.read_sql(query, conn)


,campaign,redemptions,households_redeemed
0,18,7183,214
1,13,6919,196
2,8,4092,158
3,26,803,31
4,30,704,36
5,25,671,24
6,23,660,23
7,22,517,17
8,17,495,18
9,16,473,19


In [29]:
query = """
SELECT
    c.campaign,
    COUNT(DISTINCT c.household_key) AS households_targeted,
    COUNT(DISTINCT r.household_key) AS households_redeemed,
    ROUND(
        100.0 * COUNT(DISTINCT r.household_key)
        / NULLIF(COUNT(DISTINCT c.household_key), 0),
        2
    ) AS redemption_rate_pct
FROM campaign_table c
LEFT JOIN coupon_redempt r
    ON c.campaign = r.campaign
    AND c.household_key = r.household_key
GROUP BY c.campaign
ORDER BY redemption_rate_pct DESC;
"""

campaign_rate_sql = pd.read_sql(query, conn)

campaign_rate_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\2280295941.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  campaign_rate_sql = pd.read_sql(query, conn)


,campaign,households_targeted,households_redeemed,redemption_rate_pct
0,18,1133,214,18.89
1,13,1077,196,18.20
2,3,12,2,16.67
3,8,1076,158,14.68
4,25,187,24,12.83
5,23,183,23,12.57
6,15,17,2,11.76
7,19,130,15,11.54
8,9,176,20,11.36
9,29,118,13,11.02


### 3.4.4 Campaign Sales Performance

In [30]:
query = """
SELECT
    c.campaign,
    COUNT(DISTINCT c.household_key) AS households_targeted,
    ROUND(SUM(t.sales_value), 2) AS total_sales,
    COUNT(DISTINCT t.basket_id) AS transactions
FROM campaign_table c
JOIN transaction_data t
    ON c.household_key = t.household_key
GROUP BY c.campaign
ORDER BY total_sales DESC;
"""

campaign_sales_sql = pd.read_sql(query, conn)

campaign_sales_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\1702469922.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  campaign_sales_sql = pd.read_sql(query, conn)


,campaign,households_targeted,total_sales,transactions
0,18,1133,64513035.07,196571
1,13,1077,63484361.05,195049
2,8,1076,59959606.08,182041
3,22,276,22386632.51,64198
4,20,244,20467312.47,56370
5,30,361,20041571.11,61362
6,14,224,19087131.25,50451
7,26,332,18423414.02,56234
8,25,187,18129069.86,49202
9,23,183,16968247.67,42307


### 3.4.5 Campaign Period Performance

In [31]:
query = """
SELECT
    c.campaign,
    cd.start_day,
    cd.end_day,
    COUNT(DISTINCT c.household_key) AS households_targeted,
    ROUND(SUM(t.sales_value), 2) AS campaign_period_sales,
    SUM(t.quantity) AS quantity,
    COUNT(DISTINCT t.basket_id) AS transactions
FROM campaign_table c
JOIN campaign_desc cd
    ON c.campaign = cd.campaign
JOIN transaction_data t
    ON c.household_key = t.household_key
    AND t.day BETWEEN cd.start_day AND cd.end_day
GROUP BY
    c.campaign,
    cd.start_day,
    cd.end_day
ORDER BY campaign_period_sales DESC;
"""

campaign_period_sql = pd.read_sql(query, conn)

campaign_period_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\2528368525.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  campaign_period_sql = pd.read_sql(query, conn)


,campaign,start_day,end_day,households_targeted,campaign_period_sales,quantity,transactions
0,18,587,642,1104,64525501.81,2213105972,16669
1,8,412,460,1057,51157506.51,1785824359,14359
2,13,504,551,1050,50346754.48,1956527287,14163
3,20,615,685,243,26102848.42,935346214,5719
4,14,531,596,224,20543176.72,776849403,4799
5,30,323,369,347,15482993.02,494754480,4200
6,22,624,656,270,13040954.08,475363262,3055
7,26,224,264,316,13008120.73,414176466,3495
8,11,477,523,213,12800545.23,509232251,3380
9,23,646,684,181,11845396.64,528812834,2524


### 3.4.6 Product Performance

In [32]:
query = """
SELECT
    p.product_id,
    p.department,
    p.commodity_desc,
    ROUND(SUM(t.sales_value), 2) AS sales,
    SUM(t.quantity) AS quantity,
    COUNT(DISTINCT t.basket_id) AS transactions
FROM transaction_data t
JOIN product p
    ON t.product_id = p.product_id
GROUP BY
    p.product_id,
    p.department,
    p.commodity_desc
ORDER BY sales DESC
LIMIT 20;
"""

product_sql = pd.read_sql(query, conn)

product_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\3721553410.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product_sql = pd.read_sql(query, conn)


,product_id,department,commodity_desc,sales,quantity,transactions
0,6534178,KIOSK-GAS,COUPON/MISC ITEMS,503867.11,216532156,19820
1,6533889,MISC SALES TRAN,COUPON/MISC ITEMS,46311.32,18479630,1453
2,1029743,GROCERY,FLUID MILK PRODUCTS,41072.06,16976,14430
3,6534166,MISC SALES TRAN,COUPON/MISC ITEMS,33594.48,13980451,1206
4,1082185,PRODUCE,TROPICAL FRUIT,29482.00,30896,29778
5,916122,MEAT,CHICKEN,28951.93,6985,4415
6,6533765,KIOSK-GAS,FUEL,28837.27,1756,1756
7,1106523,GROCERY,FLUID MILK PRODUCTS,27818.44,11500,9826
8,995242,GROCERY,FLUID MILK PRODUCTS,26502.46,21758,12542
9,5569230,GROCERY,SOFT DRINKS,23651.56,7682,4834


### 3.4.7 Discount & Coupon Analysis

In [33]:
query = """
SELECT
    CASE
        WHEN coupon_disc < 0 THEN 'Coupon Used'
        ELSE 'No Coupon'
    END AS coupon_status,
    COUNT(*) AS transaction_lines,
    COUNT(DISTINCT basket_id) AS transactions,
    ROUND(SUM(sales_value), 2) AS sales,
    ROUND(SUM(ABS(coupon_disc)), 2) AS coupon_discount,
    ROUND(AVG(sales_value), 2) AS avg_line_sales
FROM transaction_data
GROUP BY coupon_status
ORDER BY sales DESC;
"""

coupon_sql = pd.read_sql(query, conn)

coupon_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\514972785.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  coupon_sql = pd.read_sql(query, conn)


,coupon_status,transaction_lines,transactions,sales,coupon_discount,avg_line_sales
0,No Coupon,2559310,276153,7930058.39,0.00,3.1
1,Coupon Used,36422,16751,127404.69,42611.54,3.5


### 3.4.8 Household Value Analysis

In [34]:
query = """
SELECT
    household_key,
    ROUND(SUM(sales_value), 2) AS total_sales,
    SUM(quantity) AS quantity,
    COUNT(DISTINCT basket_id) AS transactions,
    ROUND(
        SUM(sales_value) / COUNT(DISTINCT basket_id),
        2
    ) AS avg_basket_value
FROM transaction_data
GROUP BY household_key
ORDER BY total_sales DESC
LIMIT 20;
"""

household_value_sql = pd.read_sql(query, conn)

household_value_sql

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\3628615640.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  household_value_sql = pd.read_sql(query, conn)


,household_key,total_sales,quantity,transactions,avg_basket_value
0,1023,38319.79,4479917,603,63.55
1,1609,27859.68,2146715,412,67.62
2,2322,23646.92,992718,323,73.21
3,1453,21661.29,95743,761,28.46
4,2459,20671.50,488789,971,21.29
5,1430,20352.99,1741892,344,59.17
6,718,19299.86,869732,599,32.22
7,707,19194.42,1640193,498,38.54
8,1653,19153.75,1065654,541,35.40
9,1111,18894.72,11216,321,58.86


## 3.5 SQL vs Pandas Validation

In [35]:
transaction = pd.read_csv("../Data/transaction_data.csv")

### 3.5.1 Total Sales Validation

In [36]:
pandas_total_sales = transaction["SALES_VALUE"].sum()

sql_total_sales_value = sql_total_sales.iloc[0]["total_sales"]

difference = pandas_total_sales - sql_total_sales_value

print(f"Pandas Total Sales:      {pandas_total_sales:,.2f}")
print(f"PostgreSQL Total Sales:  {sql_total_sales_value:,.2f}")
print(f"Difference:              {difference:,.2f}")
print(f"Validation:              {'MATCH' if abs(difference) < 0.01 else 'CHECK'}")

Pandas Total Sales:      8,057,463.08
PostgreSQL Total Sales:  8,057,463.08
Difference:              0.00
Validation:              MATCH


### 3.5.2 Department Performance Validation

In [37]:
department_pandas = (
    transaction
    .merge(
        product[["PRODUCT_ID", "DEPARTMENT"]],
        on="PRODUCT_ID",
        how="left"
    )
    .groupby("DEPARTMENT")
    .agg(
        sales=("SALES_VALUE", "sum"),
        quantity=("QUANTITY", "sum"),
        transactions=("BASKET_ID", "nunique")
    )
    .reset_index()
)

In [38]:
print(department_pandas.columns.tolist())
print(department_sql.columns.tolist())

['DEPARTMENT', 'sales', 'quantity', 'transactions']
['department', 'sales', 'quantity', 'transactions', 'households']


In [39]:
department_comparison = department_pandas.merge(
    department_sql,
    left_on="DEPARTMENT",
    right_on="department",
    suffixes=("_pandas", "_sql")
)

department_comparison["sales_difference"] = (
    department_comparison["sales_pandas"]
    - department_comparison["sales_sql"]
)

department_comparison["quantity_difference"] = (
    department_comparison["quantity_pandas"]
    - department_comparison["quantity_sql"]
)

department_comparison["transaction_difference"] = (
    department_comparison["transactions_pandas"]
    - department_comparison["transactions_sql"]
)

department_comparison

,DEPARTMENT,sales_pandas,quantity_pandas,transactions_pandas,department,sales_sql,quantity_sql,transactions_sql,households,sales_difference,quantity_difference,transaction_difference
0,,0.00,0,7392,,0.00,0,7392,1423,0.000000e+00,0,0
1,AUTOMOTIVE,452.66,67,65,AUTOMOTIVE,452.66,67,65,61,0.000000e+00,0,0
2,CHARITABLE CONT,7.74,3,2,CHARITABLE CONT,7.74,3,2,2,0.000000e+00,0,0
3,CHEF SHOPPE,2290.80,875,749,CHEF SHOPPE,2290.80,875,749,382,0.000000e+00,0,0
4,CNTRL/STORE SUP,49.05,22,21,CNTRL/STORE SUP,49.05,22,21,20,0.000000e+00,0,0
5,COSMETICS,32360.37,7984,5166,COSMETICS,32360.37,7984,5166,1286,0.000000e+00,0,0
6,COUP/STR & MFG,1027.72,1011,816,COUP/STR & MFG,1027.72,1011,816,442,0.000000e+00,0,0
7,DAIRY DELI,73.23,77,53,DAIRY DELI,73.23,77,53,50,0.000000e+00,0,0
8,DELI,260866.51,67026,35639,DELI,260866.51,67026,35639,2248,0.000000e+00,0,0
9,DELI/SNACK BAR,36.08,12,11,DELI/SNACK BAR,36.08,12,11,10,0.000000e+00,0,0


### 3.5.3 Household Performance Validation

In [40]:
household_pandas = (
    transaction
    .groupby("household_key")
    .agg(
        sales=("SALES_VALUE", "sum"),
        quantity=("QUANTITY", "sum"),
        transactions=("BASKET_ID", "nunique")
    )
    .reset_index()
)

household_pandas["sales_per_transaction"] = (
    household_pandas["sales"]
    / household_pandas["transactions"]
)

In [41]:
household_comparison = household_pandas.merge(
    household_sql,
    on="household_key",
    suffixes=("_pandas", "_sql")
)

household_comparison["sales_difference"] = (
    household_comparison["sales_pandas"]
    - household_comparison["sales_sql"]
)

household_comparison["quantity_difference"] = (
    household_comparison["quantity_pandas"]
    - household_comparison["quantity_sql"]
)

household_comparison["transaction_difference"] = (
    household_comparison["transactions_pandas"]
    - household_comparison["transactions_sql"]
)

household_comparison.head(10)

,household_key,sales_pandas,quantity_pandas,transactions_pandas,sales_per_transaction_pandas,sales_sql,transactions_sql,quantity_sql,sales_per_transaction_sql,sales_difference,quantity_difference,transaction_difference
0,1,4330.16,1997,86,50.350698,4330.16,86,1997,50.35,0.000000e+00,0,0
1,2,1954.34,834,45,43.429778,1954.34,45,834,43.43,0.000000e+00,0,0
2,3,2653.21,8540,47,56.451277,2653.21,47,8540,56.45,0.000000e+00,0,0
3,4,1200.11,382,30,40.003667,1200.11,30,382,40.00,2.273737e-13,0,0
4,5,779.06,245,40,19.476500,779.06,40,245,19.48,1.136868e-13,0,0
5,6,5996.16,94690,250,23.984640,5996.16,250,94690,23.98,0.000000e+00,0,0
6,7,3400.05,1554,59,57.627966,3400.05,59,1554,57.63,0.000000e+00,0,0
7,8,5534.97,68333,113,48.982035,5534.97,113,68333,48.98,0.000000e+00,0,0
8,9,797.42,353,20,39.871000,797.42,20,353,39.87,0.000000e+00,0,0
9,10,234.34,108,9,26.037778,234.34,9,108,26.04,0.000000e+00,0,0


### 3.5.4 Campaign Performance Validation

In [42]:
campaign_table = pd.read_csv("../Data/campaign_table.csv")

campaign_pandas = (
    campaign_table
    .groupby("CAMPAIGN")
    .agg(
        households_targeted=("household_key", "nunique")
    )
    .reset_index()
    .rename(columns={"CAMPAIGN": "campaign"})
)

campaign_pandas

,campaign,households_targeted
0,1,13
1,2,48
2,3,12
3,4,81
4,5,166
5,6,65
6,7,198
7,8,1076
8,9,176
9,10,123


## 3.5.5 Final SQL vs Pandas Validation

In [43]:
#pandas- side
pandas_campaign = (
    campaign_table
    .groupby("CAMPAIGN")
    .agg(households=("household_key", "nunique"))
    .reset_index()
)

#sql- side
query = """
SELECT campaign, COUNT(DISTINCT household_key) AS households
FROM campaign_table
GROUP BY campaign;
"""
sql_campaign = pd.read_sql(query, conn)

campaign_comparison = pandas_campaign.merge(
    sql_campaign,
    left_on="CAMPAIGN", right_on="campaign",
    suffixes=("_pandas", "_sql")
)
campaign_comparison["difference"] = (
    campaign_comparison["households_pandas"] - campaign_comparison["households_sql"]
)
campaign_comparison

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\226055198.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_campaign = pd.read_sql(query, conn)


,CAMPAIGN,households_pandas,campaign,households_sql,difference
0,1,13,1,13,0
1,2,48,2,48,0
2,3,12,3,12,0
3,4,81,4,81,0
4,5,166,5,166,0
5,6,65,6,65,0
6,7,198,7,198,0
7,8,1076,8,1076,0
8,9,176,9,176,0
9,10,123,10,123,0


## 3.6 Promotional Opportunity Scoring

### 3.6.1 Household-Level Metrics and Value Segmentation

In [44]:
query = """
WITH household_base AS (
    SELECT
        household_key,
        SUM(sales_value) AS sales,
        SUM(quantity) AS quantity,
        COUNT(DISTINCT basket_id) AS transactions,
        COUNT(DISTINCT product_id) AS purchase_breadth
    FROM transaction_data
    GROUP BY household_key
)
SELECT * FROM household_base;
"""
household_base_sql = pd.read_sql(query, conn)
household_base_sql.shape

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\77777967.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  household_base_sql = pd.read_sql(query, conn)


(2500, 5)

### 3.6.2 Building the Promotional Opportunity Score

In [45]:
query = """
WITH household_base AS (
    SELECT
        household_key,
        SUM(sales_value) AS sales,
        SUM(quantity) AS quantity,
        COUNT(DISTINCT basket_id) AS transactions,
        COUNT(DISTINCT product_id) AS purchase_breadth
    FROM transaction_data
    GROUP BY household_key
),

household_medians AS (
    SELECT
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sales) AS sales_median,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY transactions) AS transactions_median
    FROM household_base
),

-- Engagement score: plain percentile rank across ALL households
household_engagement AS (
    SELECT
        household_key,
        PERCENT_RANK() OVER (ORDER BY sales) * 100 AS sales_score,
        PERCENT_RANK() OVER (ORDER BY transactions) * 100 AS transaction_score,
        PERCENT_RANK() OVER (ORDER BY purchase_breadth) * 100 AS breadth_score
    FROM household_base
),
household_engagement_scored AS (
    SELECT
        household_key,
        0.40 * sales_score + 0.30 * transaction_score + 0.30 * breadth_score AS engagement_score
    FROM household_engagement
),

-- Household-department level: artifact exclusion applied here
household_dept AS (
    SELECT
        t.household_key,
        p.department,
        SUM(t.sales_value) AS sales,
        COUNT(DISTINCT t.basket_id) AS transactions,
        SUM(t.quantity) AS quantity,
        COUNT(DISTINCT t.product_id) AS products
    FROM transaction_data t
    JOIN product p ON t.product_id = p.product_id
    WHERE p.department NOT IN ('MISC SALES TRAN', 'KIOSK-GAS', 'MISC. TRANS.')
    GROUP BY t.household_key, p.department
),

-- Department-count floor: same fix as notebook 2 cell 101
eligible_departments AS (
    SELECT department
    FROM household_dept
    GROUP BY department
    HAVING COUNT(DISTINCT household_key) >= 100
),

-- Category affinity: ranked WITHIN each department, not pooled
household_dept_scored AS (
    SELECT
        hd.household_key,
        hd.department,
        PERCENT_RANK() OVER (PARTITION BY hd.department ORDER BY hd.sales) * 100 AS sales_score,
        PERCENT_RANK() OVER (PARTITION BY hd.department ORDER BY hd.transactions) * 100 AS transaction_score,
        PERCENT_RANK() OVER (PARTITION BY hd.department ORDER BY hd.quantity) * 100 AS quantity_score,
        PERCENT_RANK() OVER (PARTITION BY hd.department ORDER BY hd.products) * 100 AS breadth_score
    FROM household_dept hd
    JOIN eligible_departments ed ON hd.department = ed.department
),
household_dept_affinity AS (
    SELECT
        household_key,
        department,
        0.30 * sales_score + 0.25 * transaction_score
            + 0.25 * quantity_score + 0.20 * breadth_score AS category_affinity_score
    FROM household_dept_scored
),

-- Each household's single best (highest-affinity) category
best_affinity AS (
    SELECT DISTINCT ON (household_key)
        household_key,
        department AS top_affinity_category,
        category_affinity_score
    FROM household_dept_affinity
    ORDER BY household_key, category_affinity_score DESC
),

-- Campaign responsiveness with reliability adjustment
campaign_scored AS (
    SELECT
        c.household_key,
        COUNT(DISTINCT c.campaign) AS campaigns_targeted,
        COUNT(DISTINCT r.campaign) AS campaigns_redeemed,
        (COUNT(DISTINCT r.campaign)::float / NULLIF(COUNT(DISTINCT c.campaign), 0)) * 100
            * (COUNT(DISTINCT c.campaign)::float / (COUNT(DISTINCT c.campaign) + 5)) AS campaign_score
    FROM campaign_table c
    LEFT JOIN coupon_redempt r
        ON c.household_key = r.household_key AND c.campaign = r.campaign
    GROUP BY c.household_key
),

-- Assemble full household score table
household_score_data AS (
    SELECT
        hb.household_key,
        hb.sales,
        hb.transactions,
        CASE
            WHEN hb.sales >= m.sales_median AND hb.transactions >= m.transactions_median
                THEN 'High Value - Frequent'
            WHEN hb.sales >= m.sales_median
                THEN 'High Value - Less Frequent'
            WHEN hb.transactions >= m.transactions_median
                THEN 'Low Value - Frequent'
            ELSE 'Low Value'
        END AS value_segment,
        COALESCE(cs.campaign_score, 0) AS campaign_score,
        he.engagement_score,
        COALESCE(ba.top_affinity_category, 'None') AS top_affinity_category,
        COALESCE(ba.category_affinity_score, 0) AS category_affinity_score
    FROM household_base hb
    CROSS JOIN household_medians m
    LEFT JOIN campaign_scored cs ON hb.household_key = cs.household_key
    LEFT JOIN household_engagement_scored he ON hb.household_key = he.household_key
    LEFT JOIN best_affinity ba ON hb.household_key = ba.household_key
),

-- Re-rank all three components to comparable percentile scales.
-- campaign_score_pct: zero group scores 0 explicitly; only non-zero
-- households are ranked against each other (the zero-inflation fix).
household_score_pct AS (
    SELECT
        *,
        CASE WHEN campaign_score = 0 THEN 0
            ELSE PERCENT_RANK() OVER (
                PARTITION BY (campaign_score = 0) ORDER BY campaign_score
             ) * 100
        END AS campaign_score_pct,
        PERCENT_RANK() OVER (ORDER BY category_affinity_score) * 100 AS category_affinity_pct,
        PERCENT_RANK() OVER (ORDER BY engagement_score) * 100 AS engagement_pct
    FROM household_score_data
)

-- Final promotional opportunity score: engagement 35%, category affinity 30%, campaign 35%
SELECT
    household_key,
    sales,
    value_segment,
    top_affinity_category,
    ROUND(campaign_score_pct::numeric, 2) AS campaign_score_pct,
    ROUND(category_affinity_pct::numeric, 2) AS category_affinity_pct,
    ROUND(engagement_pct::numeric, 2) AS engagement_pct,
    ROUND(
        (0.35 * engagement_pct + 0.30 * category_affinity_pct + 0.35 * campaign_score_pct)::numeric,
        2
    ) AS promotional_opportunity_score
FROM household_score_pct
ORDER BY promotional_opportunity_score DESC;
"""
household_score_sql = pd.read_sql(query, conn)
household_score_sql.head(20)

C:\Users\Saksham\AppData\Local\Temp\ipykernel_14392\1194878036.py:162: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  household_score_sql = pd.read_sql(query, conn)


,household_key,sales,value_segment,top_affinity_category,campaign_score_pct,category_affinity_pct,engagement_pct,promotional_opportunity_score
0,979,12854.66,High Value - Frequent,NUTRITION,96.77,97.84,96.72,97.07
1,2280,12332.91,High Value - Frequent,COSMETICS,94.69,99.64,96.44,96.79
2,389,14880.92,High Value - Frequent,PRODUCE,94.92,96.56,97.88,96.45
3,982,18790.34,High Value - Frequent,DRUG GM,93.07,95.88,99.16,96.04
4,1823,12268.85,High Value - Frequent,DELI,97.46,91.32,97.76,95.72
5,1633,10162.78,High Value - Frequent,DELI,91.69,97.48,95.68,94.82
6,115,13262.10,High Value - Frequent,SEAFOOD-PCKGD,94.92,93.72,95.08,94.61
7,1098,9500.70,High Value - Frequent,SPIRITS,90.99,97.28,95.84,94.57
8,13,13190.92,High Value - Frequent,DRUG GM,98.38,89.24,95.16,94.51
9,1726,8131.91,High Value - Frequent,NUTRITION,97.46,95.28,90.32,94.30


### 3.6.3: Validation Against the Pandas Scoring Results


In [46]:
household_score_sql[household_score_sql["household_key"] == 2459]

,household_key,sales,value_segment,top_affinity_category,campaign_score_pct,category_affinity_pct,engagement_pct,promotional_opportunity_score
228,2459,20671.5,High Value - Frequent,GROCERY,0.0,99.76,100.0,64.93


### 3.6.4: Identifying Never-Targeted Households

In [47]:
never_targeted_sql = household_score_sql[household_score_sql["campaign_score_pct"] == 0]
never_targeted_sql.sort_values("promotional_opportunity_score", ascending=False).head(10)

,household_key,sales,value_segment,top_affinity_category,campaign_score_pct,category_affinity_pct,engagement_pct,promotional_opportunity_score
228,2459,20671.50,High Value - Frequent,GROCERY,0.0,99.76,100.00,64.93
229,1023,38319.79,High Value - Frequent,GARDEN CENTER,0.0,99.92,99.76,64.89
236,900,16450.53,High Value - Frequent,DELI,0.0,99.28,99.72,64.69
238,1489,17251.53,High Value - Frequent,GROCERY,0.0,98.48,99.84,64.49
239,707,19194.42,High Value - Frequent,PASTRY,0.0,98.32,99.80,64.43
244,909,12414.56,High Value - Frequent,FLORAL,0.0,97.60,99.64,64.15
245,1510,11211.37,High Value - Frequent,SALAD BAR,0.0,98.76,98.60,64.14
246,328,17332.13,High Value - Frequent,DELI,0.0,99.56,97.92,64.14
247,591,13534.93,High Value - Frequent,MEAT,0.0,98.92,98.32,64.09
248,2312,14682.81,High Value - Frequent,SALAD BAR,0.0,99.08,98.12,64.07


### 3.6.5: Promotional Opportunity by Value Segment

In [48]:
segment_summary_sql = (
    household_score_sql
    .groupby("value_segment")
    .agg(
        households=("household_key", "count"),
        avg_opportunity_score=("promotional_opportunity_score", "mean"),
        total_sales=("sales", "sum")
    )
)
segment_summary_sql

,households,avg_opportunity_score,total_sales
value_segment,,,
High Value - Frequent,1050,55.572371,6152810.71
High Value - Less Frequent,200,39.242800,642949.32
Low Value,1045,15.256871,935697.06
Low Value - Frequent,205,31.384780,326005.99


In [49]:
# Power BI export: household segments

household_segments = (
    segment_summary_sql
    .reset_index()
    .rename(columns={
        "households": "households",
        "avg_opportunity_score": "avg_opportunity_score",
        "total_sales": "total_sales"
    })
)

household_segments["sales_share_pct"] = (
    household_segments["total_sales"]
    / household_segments["total_sales"].sum()
    * 100
).round(2)

household_segments.to_csv(
    "household_segments.csv",
    index=False
)

household_segments

,value_segment,households,avg_opportunity_score,total_sales,sales_share_pct
0,High Value - Frequent,1050,55.572371,6152810.71,76.36
1,High Value - Less Frequent,200,39.242800,642949.32,7.98
2,Low Value,1045,15.256871,935697.06,11.61
3,Low Value - Frequent,205,31.384780,326005.99,4.05


In [50]:
# Power BI export: category performance

category_performance = (
    department_sql
    .reset_index()
)

category_performance["sales_share_pct"] = (
    category_performance["sales"]
    / category_performance["sales"].sum()
    * 100
).round(2)

category_performance.to_csv(
    "category_performance.csv",
    index=False
)

category_performance

,index,department,sales,quantity,transactions,households,sales_share_pct
0,0,GROCERY,4093814.14,2194762,215411,2500,50.81
1,1,DRUG GM,1055358.03,353844,118195,2491,13.10
2,2,PRODUCE,557452.11,319993,89026,2456,6.92
3,3,MEAT,548786.81,119113,53401,2351,6.81
4,4,KIOSK-GAS,544222.28,221254887,22056,1371,6.75
5,5,MEAT-PCKGD,412436.77,148148,58466,2392,5.12
6,6,DELI,260866.51,67026,35639,2248,3.24
7,7,PASTRY,121739.86,49820,30099,2281,1.51
8,8,MISC SALES TRAN,119960.04,36080860,5992,1393,1.49
9,9,NUTRITION,97669.04,43253,16820,1719,1.21


In [51]:
# Power BI export: top 10 promotional opportunities

promotional_opportunities = (
    never_targeted_sql
    .sort_values("promotional_opportunity_score", ascending=False)
    .head(10)
    .reset_index()
)

promotional_opportunities.to_csv(
    "promotional_opportunities.csv",
    index=False
)

promotional_opportunities

,index,household_key,sales,value_segment,top_affinity_category,campaign_score_pct,category_affinity_pct,engagement_pct,promotional_opportunity_score
0,228,2459,20671.50,High Value - Frequent,GROCERY,0.0,99.76,100.00,64.93
1,229,1023,38319.79,High Value - Frequent,GARDEN CENTER,0.0,99.92,99.76,64.89
2,236,900,16450.53,High Value - Frequent,DELI,0.0,99.28,99.72,64.69
3,238,1489,17251.53,High Value - Frequent,GROCERY,0.0,98.48,99.84,64.49
4,239,707,19194.42,High Value - Frequent,PASTRY,0.0,98.32,99.80,64.43
5,244,909,12414.56,High Value - Frequent,FLORAL,0.0,97.60,99.64,64.15
6,245,1510,11211.37,High Value - Frequent,SALAD BAR,0.0,98.76,98.60,64.14
7,246,328,17332.13,High Value - Frequent,DELI,0.0,99.56,97.92,64.14
8,247,591,13534.93,High Value - Frequent,MEAT,0.0,98.92,98.32,64.09
9,248,2312,14682.81,High Value - Frequent,SALAD BAR,0.0,99.08,98.12,64.07
